# Setup

In [1]:
# user setup
import os

# custom Kaggle login (environment variables are recommended)
os.environ["KAGGLE_USERNAME"] = "" or os.getenv("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = "" or os.getenv("KAGGLE_KEY")

# spark configuration
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64" or os.getenv("JAVA_HOME")
os.environ["SPARK_HOME"] = "" or os.getenv("SPARK_HOME")

# subsampling: percentage of commenting users to keep
USERS_PERCENTAGE = 0.1
assert 0 < USERS_PERCENTAGE <= 1, "invalid USER_PERCENTAGE, should be in (0, 1]"

# configuration
MIN_KW_FREQ = 5 # minimum frequency of keywords to be included in the profile (for the dictionary approach)
FEATURES_SIZE = 5000 # number of features, buckets of the hash (for the hashing approach)
LSH_NUM_HASHES = 100 # number of random hyperplanes (signatures/sketches)
LSH_BANDS = 10 # TODO: play with b and r (calculate the threshold for different values)
LSH_ROWS = 10
LSH_MAX_BUCKET_SIZE = 500 # maximum number of users/articles in the same bucket
RECOMMENDED_ARTICLES = 10 # number of articles to recommend to each user
assert LSH_NUM_HASHES == LSH_BANDS * LSH_ROWS, "invalid LSH_NUM_HASHES, must be equal to LSH_BANDS * LSH_ROWS"


In [2]:
# setup dataset
%pip install kaggle
!test -d dataset && echo "Skipping dataset download" || (kaggle datasets download -d "benjaminawd/new-york-times-articles-comments-2020" && unzip -d dataset new-york-times-articles-comments-2020.zip && rm -f new-york-times-articles-comments-2020.zip 2> /dev/null)


Note: you may need to restart the kernel to use updated packages.
Skipping dataset download


In [3]:
# setup spark
!test -d spark && echo "Skipping spark download" || (wget -q https://dlcdn.apache.org/spark/spark-4.1.1/spark-4.1.1-bin-hadoop3.tgz -O spark.tgz && mkdir spark && tar xvf spark.tgz -C spark --strip-components=1 > /dev/null && rm spark.tgz)
%pip install pyspark
%pip install py4j
import pyspark
ss = (pyspark.sql.SparkSession
    .builder
    .getOrCreate())
sc = ss.sparkContext


Skipping spark download
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
csv_options = {
    "header": "true",
    "inferSchema": "true",
    "quote": '"',
    "escape": '"', # quotes " in the dataset are escaped ""
    "multiLine": "true",
    "mode": "PERMISSIVE",
}

full_articles = ss.read.options(**csv_options).csv("dataset/nyt-articles-2020.csv")
full_comments = ss.read.options(**csv_options).csv("dataset/nyt-comments-part0.csv") # TODO: this is a subset of all comments

print(f"Number of articles: {full_articles.count()}")
print(f"Number of comments: {full_comments.count()}")
full_articles.take(1), full_comments.take(1)


Number of articles: 16787
Number of comments: 500000


([Row(newsdesk='Editorial', section='Opinion', subsection=None, material='Editorial', headline='Protect Veterans From Fraud', abstract='Congress could do much more to protect Americans who have served their country from predatory for-profit colleges.', keywords="['Veterans', 'For-Profit Schools', 'Financial Aid (Education)', 'Frauds and Swindling', 'Colleges and Universities', 'Veterans Affairs Department', 'Federal Trade Commission', 'University of Phoenix', 'Career Education Corporation']", word_count=680, pub_date=datetime.datetime(2020, 1, 1, 0, 18, 54), n_comments=186, uniqueID='nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd')],
 [Row(commentID=104387472, status='approved', commentSequence=104387472, userID=60215558, userDisplayName='magicisnotreal', userLocation='earth', userTitle=None, commentBody='Here is something I think is fraudulent that vets are subject to\n If you use your VA home loan option you have to pay higher interest rates regardless of your credit rating becua

In [5]:
%pip install mmh3 # non-cryptographic hash, better distribution
# utility hash function: bytes | str -> signed int 32
def h(data):
    import mmh3
    return mmh3.hash(data)


Note: you may need to restart the kernel to use updated packages.


## RDD

In [6]:
# parse keywords column utility
def parse_keywords(row):
    import re
    if not row["keywords"]: return []
    kws = re.sub(r'\[|\]|\'|\"', '', row["keywords"].lower()).strip()
    if not kws: return []
    return [w.strip() for w in kws.split(",") if len(w.strip()) > 0]

# keep only relevant information for recommendations porpouse, convert from dataframe to RDD tuples
articles = (full_articles
            .select("uniqueID", "newsdesk", "section", "subsection", "keywords")
            .rdd
            .map(lambda row: (row["uniqueID"], row["newsdesk"], row["section"], row["subsection"], parse_keywords(row))))
comments = (full_comments
            .select("articleID", "userID")
            .rdd
            .map(lambda row: (row["articleID"], row["userID"])))
users = comments.map(lambda x: x[1]).distinct()

print(f"Number of articles: {articles.count()}")
print(f"Number of comments: {comments.count()}")
print(f"Number of users: {users.count()}")
articles.take(1), comments.take(1)


Number of articles: 16787
Number of comments: 500000
Number of users: 91437


([('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd',
   'Editorial',
   'Opinion',
   None,
   ['veterans',
    'for-profit schools',
    'financial aid (education)',
    'frauds and swindling',
    'colleges and universities',
    'veterans affairs department',
    'federal trade commission',
    'university of phoenix',
    'career education corporation'])],
 [('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 60215558)])

# Building Profiles

## Utility Matrix

In [7]:
utility_matrix = comments.distinct() # ignore multiple comments from same user on same article
print(f"Number of entries in sparse utility matrix: {utility_matrix.count()}")
utility_matrix.take(5)


Number of entries in sparse utility matrix: 356283


[('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 60215558),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 65691034),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 65110053),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 66232009),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', 73299044)]

## Articles Profile - via dictionary

In [8]:
# count frequencies
keywords = (articles
            .flatMap(lambda row: [(k, 1) for k in row[4]])
            .reduceByKey(lambda a, b: a + b))

# prune too rare keywords
pruned_keywords = (keywords
                   .filter(lambda x: x[1] >= MIN_KW_FREQ and len(x[0]) <= 100)
                   .map(lambda x: f"KEY {x[0]}")
                   .collect())

# extract all features
features = sorted(pruned_keywords + (articles
            .flatMap(lambda row: [f"{f} {row[i+1].lower().strip()}" for i, f in enumerate(["NEW", "SEC", "SUB"]) if row[i+1]])
            .filter(lambda x: len(x) < 100)
            .distinct()
            .collect()))

# features mapping dictionary, needs to be broadcasted to every node
features_mapping = {f: i for i, f in enumerate(features)}
print(f"Extacted {len(features_mapping)} features (broadcasting at most {((100 * 4) + 4) * len(features_mapping) // 1024} KB)")
broadcast_dict = sc.broadcast(features_mapping)

# build sparse vectors for profiles (article_id, feature_index)
def build_article_profile_dict(row):
    mapping = broadcast_dict.value
    res = []
    for i, f in enumerate(["NEW", "SEC", "SUB"]):
        if not row[i+1]: continue
        f = f"{f} {row[i+1].lower().strip()}"
        if f not in mapping: continue
        res.append((row[0], mapping[f]))
    for k in row[4]:
        k = f"KEY {k.lower().strip()}"
        if k not in mapping: continue
        res.append((row[0], mapping[k]))
    return res

articles_profile_dict = (articles
                         .flatMap(build_article_profile_dict) # (article_id, feature_index)
                         .map(lambda x: (x[0], (x[1], 1)))) # (article_id, (feature_index, frequency = 1))
print(f"Number of entries in sparse articles profile via dictionary: {articles_profile_dict.count()}")
articles_profile_dict.take(10)


Extacted 3555 features (broadcasting at most 1402 KB)
Number of entries in sparse articles profile via dictionary: 158079


[('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (3395, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (3465, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (3193, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (1131, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (1097, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (1158, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (628, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (3194, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (1086, 1)),
 ('nyt://article/9edddb54-0aa3-5835-a833-d311a76f1e7c', (3399, 1))]

## Articles Profile - via hashing

In [9]:
# build sparse vectors for profiles (article_id, feature_hash)
def build_article_profile_hash(row):
    res = []
    for i, f in enumerate(["NEW", "SEC", "SUB"]):
        if not row[i+1]: continue
        f = f"{f} {row[i+1].lower().strip()}"
        res.append((row[0], h(f) % FEATURES_SIZE))
    for k in row[4]:
        k = f"KEY {k.lower().strip()}"
        res.append((row[0], h(k) % FEATURES_SIZE))
    return res

articles_profile_hash = (articles
                         .flatMap(build_article_profile_hash) # (article_id, feature_hash)
                         .map(lambda x: (x[0], (x[1], 1)))) # (article_id, (feature_hash, frequency = 1))
print(f"Number of entries in sparse articles profile via hash: {articles_profile_hash.count()}")
articles_profile_hash.take(10)


Number of entries in sparse articles profile via hash: 185396


[('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (653, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (2704, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (2692, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (1483, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (4497, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (260, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (1675, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (2683, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (3196, 1)),
 ('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd', (598, 1))]

## WARNING

From now on, the operations will be **independent** of how the profiles were computed (via dictionary or hashing).
Change the content of next cell to change approach: (`articles_profile_dict` or `articles_profile_hash`)

In [10]:
articles_profile = articles_profile_hash


## Users Profile

In [11]:
# build user profiles by joining utility matrix with article profiles, then counting feature frequencies for each user
users_profile = (utility_matrix
                 .join(articles_profile) # (article_id, (user_id, (feature_index, frequency = 1)))
                 .map(lambda x: ((x[1][0], x[1][1][0]), 1)) # ((user_id, feature_index), 1)
                 .reduceByKey(lambda a, b: a + b) # ((user_id, feature_index), frequency)
                 .map(lambda x: (x[0][0], (x[0][1], x[1])))) # (user_id, (feature_index, frequency))
print(f"Number of entries in sparse users profile: {users_profile.count()}")
users_profile.take(10)


Number of entries in sparse users profile: 2801463


[(5646097, (535, 36)),
 (5646097, (2881, 36)),
 (5646097, (4221, 36)),
 (86273619, (535, 11)),
 (86273619, (2881, 11)),
 (86273619, (4221, 11)),
 (61536727, (535, 9)),
 (61536727, (2881, 9)),
 (61536727, (4221, 9)),
 (41508509, (535, 20))]

# Computing Similarity

## Profiles sketches (signatures)

In [12]:
import random

# random vectors of +1 and -1 of dimensione FEATURES_SIZE
random_hyperplanes = [[random.choice([-1, 1]) for _ in range(FEATURES_SIZE)] for _ in range(LSH_NUM_HASHES)]
b_hyperplanes = sc.broadcast(random_hyperplanes)
print(f"Generated {LSH_NUM_HASHES} random hyperplanes (broadcasting at most {(LSH_NUM_HASHES * FEATURES_SIZE * 4) // 1024} KB)")

def compute_sketch(row):
    item_id, sparse_features = row
    sketch = []
    for plane in b_hyperplanes.value:
        # dot product: if above or below the hyperplane
        # only considering the features that exist in the sparse vector
        dot_product = sum(weight * plane[feat] for feat, weight in sparse_features)
        # 1 if positive, 0 if negative
        sketch.append(1 if dot_product > 0 else 0)
    return (item_id, sketch)

# compute sketches ("equivalent" of signature) for users and articles
users_sketches = (users_profile # (user_id, (feature_index, frequency))
                  .groupByKey()
                  .mapValues(list) # (user_id, [(feature_index, frequency), ...])
                  .map(compute_sketch)) # (user_id, sketch)
articles_sketches = (articles_profile # (article_id, (feature_index, frequency = 1))
                  .groupByKey()
                  .mapValues(list) # (article_id, [(feature_index, frequency), ...])
                  .map(compute_sketch)) # (article_id, sketch)

print(f"Number of users sketches: {users_sketches.count()}")
print(f"Number of articles sketches: {articles_sketches.count()}")
users_sketches.take(1), articles_sketches.take(1)


Generated 100 random hyperplanes (broadcasting at most 1953 KB)
Number of users sketches: 91437
Number of articles sketches: 16787


([(13120442,
   [1,
    1,
    1,
    0,
    1,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    1,
    0,
    0,
    0,
    0,
    1,
    1,
    0,
    1,
    0,
    0,
    0,
    0,
    0,
    1,
    0,
    0,
    0,
    0,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    1,
    1,
    0,
    0,
    1,
    1,
    0,
    1,
    0,
    1,
    0,
    0,
    1,
    0,
    1,
    1,
    0,
    0,
    1,
    1,
    1,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    1,
    1,
    0,
    0,
    0,
    0,
    1,
    0,
    1])],
 [('nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd',
   [1,
    1,
    1,
    0,
    1,
    0,
    0,
    1,
    0,
    0,
    1,
    0,
    1,
    1,
    0,
    0,
    1,
    0,
    0,
    0,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0,
    0,
    1,
    1,
   

## LSH buckets

In [13]:
def lsh_buckets(row):
    item_id, sketch = row
    res = []
    for band in range(LSH_BANDS):
        band_signature = tuple(sketch[band*LSH_ROWS : (band+1)*LSH_ROWS])
        bucket_hash = h(f"b{band}_{band_signature}") # include band index to avoid collisions between different bands
        res.append((bucket_hash, item_id))
    return res

users_buckets = users_sketches.flatMap(lsh_buckets) # (bucket_hash, user_id)
articles_buckets = articles_sketches.flatMap(lsh_buckets) # (bucket_hash, article_id)

print(f"Total number of users buckets: {users_buckets.count()}")
print(f"Total number of articles buckets: {articles_buckets.count()}")

# buckets too big are not useful, too few information
valid_users_buckets = (users_buckets
                       .map(lambda x: (x[0], 1))
                       .reduceByKey(lambda a, b: a + b)
                       .filter(lambda x: x[1] <= LSH_MAX_BUCKET_SIZE))

valid_articles_buckets = (articles_buckets
                          .map(lambda x: (x[0], 1))
                          .reduceByKey(lambda a, b: a + b)
                          .filter(lambda x: x[1] <= LSH_MAX_BUCKET_SIZE))

# the join works as an intersection between keys
users_buckets = (users_buckets # (bucket_hash, user_id)
                 .join(valid_users_buckets) # (bucket_hash, (user_id, count))
                 .map(lambda x: (x[0], x[1][0]))) # (bucket_hash, user_id)
articles_buckets = (articles_buckets # (bucket_hash, article_id)
                    .join(valid_articles_buckets) # (bucket_hash, (article_id, count))
                    .map(lambda x: (x[0], x[1][0]))) # (bucket_hash, article_id)

print(f"Valid (less than {LSH_MAX_BUCKET_SIZE} users) number of users buckets: {users_buckets.count()} ")
print(f"Valid (less than {LSH_MAX_BUCKET_SIZE} articles) number of articles buckets: {articles_buckets.count()} ")

users_buckets.take(10), articles_buckets.take(10)


Total number of users buckets: 914370
Total number of articles buckets: 167870
Valid (less than 500 users) number of users buckets: 555221 
Valid (less than 500 articles) number of articles buckets: 161096 


([(-1930329068, 13120442),
  (-1930329068, 55563058),
  (-1930329068, 71482858),
  (-1930329068, 17091468),
  (-1930329068, 58102490),
  (-1930329068, 78844498),
  (-1930329068, 65225626),
  (-1930329068, 28863574),
  (-1930329068, 77867156),
  (-1930329068, 33784802)],
 [(1952587536, 'nyt://article/69a7090b-9f36-569e-b5ab-b0ba5bb3ccbd'),
  (1952587536, 'nyt://article/556015ab-fc1b-55c5-9ef7-61573f21e31a'),
  (1952587536, 'nyt://article/e52c07fb-2366-5ab0-92d6-4591459009d2'),
  (1952587536, 'nyt://article/3dbb7a80-db4b-50fa-a52a-9366722d8d86'),
  (1952587536, 'nyt://article/eedf8c65-6afb-5af8-9aac-c423aa63c64c'),
  (1952587536, 'nyt://interactive/dbcb806a-7b2e-53ec-a7bc-1143c97f95ae'),
  (1952587536, 'nyt://article/9b36dc64-6f13-5b47-931f-cd74b31bb651'),
  (1952587536, 'nyt://article/a93604d9-1953-5af3-b868-2c05dc212e62'),
  (1952587536, 'nyt://article/72946ab5-9fb3-5aea-b275-ceb9b6a4ca5b'),
  (1952587536, 'nyt://article/a2f74cd4-aa9e-5634-9e95-4c276be11b67')])

## LSH filter

In [14]:
#           BUCKET
# APPROACH  PRUNING  DISTINCT  SUBTRACT  BANDS  ROWS  RESULTS  TIME
#   hash      no        no       no       20     5     ~1.3b  3min 20s
#   hash      no        no       no       5      20    ~13m        13s
#   hash      no        no       no       10     10    ~60m        20s
#   hash      no       yes       no       10     10    ~35m   5min 40s
#   hash     yes        no       no       10     10    ~13m         2s
#   hash     yes       yes       no       10     10    ~12m        20s
#   hash     yes       yes      yes       10     10    ~12m   4min

lsh_recommendations = (users_buckets # (bucket_hash, user_id)
                       .join(articles_buckets) # (bucket_hash, (user_id, article_id))
                       .map(lambda x: (x[1][0], x[1][1])) # (user_id, article_id)
                       .distinct() # remove matches from multiple bands
                       .subtract(utility_matrix.map(lambda x: (x[1], x[0])))) # remove already commented articles

# TODO: works but has the issue of grouping the full user interactions and full recommentations
# lsh_recommendations = (users_buckets # (bucket_hash, user_id)
#                        .join(articles_buckets) # (bucket_hash, (user_id, article_id))
#                        .map(lambda x: (x[1][0], x[1][1])) # (user_id, article_id)
#                        .groupByKey()
#                        .mapValues(set)
#                        .join(utility_matrix.map(lambda x: (x[1], x[0])).groupByKey().mapValues(set))
#                        .mapValues(lambda x: x[0] - x[1])
#                        .flatMap(lambda x: [(x[0], a) for a in x[1]])) # (user_id, article_id)

print(f"Number of recommendations that passed LSH filter: {lsh_recommendations.count()}")
lsh_recommendations.take(10)


Number of recommendations that passed LSH filter: 12168688


[(37921454, 'nyt://interactive/97abd07d-886e-5130-98cb-96657e4f24d7'),
 (21516832, 'nyt://article/91149d7d-49cc-54a8-b8dd-4643fe5cd04a'),
 (80821286, 'nyt://article/e28937ba-1359-52bb-b275-205c4d88bc8c'),
 (89553397, 'nyt://article/1f6549f8-e079-5cbc-81e9-7779a6324724'),
 (63456544, 'nyt://article/d1b6006b-e29e-5290-8217-33e30d357f35'),
 (67787919, 'nyt://article/6fc780b9-e532-5865-b9c4-3f0a565aec2e'),
 (63782, 'nyt://article/1cb618f9-e5bc-59e4-be3b-b700747394e0'),
 (36995974, 'nyt://article/bc141064-ee87-5aff-93c0-fe9a09728745'),
 (19510297, 'nyt://article/6401c005-15f5-5808-a713-7e2946162677'),
 (72983564, 'nyt://article/19dc0509-6d5f-5c8d-8d1d-508d4118a4a2')]

## Actual Cosine Similarity

In [15]:
# cosine similarity between sparse profiles, user_profile and article_profile should be dicts {feature_index: weight}
def cosine_similarity(user_profile, article_profile):
    dot_product = sum(user_profile.get(feat, 0) * weight for feat, weight in article_profile.items())
    user_norm = sum(weight ** 2 for weight in user_profile.values()) ** 0.5
    article_norm = sum(weight ** 2 for weight in article_profile.values()) ** 0.5
    if user_norm == 0 or article_norm == 0:
        return 0.0
    return dot_product / (user_norm * article_norm)

# calculate actual similarity by joining the profiles and applying
recommendations = (lsh_recommendations # (user_id, article_id)
                   .join(users_profile.groupByKey().mapValues(dict)) # (user_id, (article_id, {profile}))
                   .map(lambda x: (x[1][0], (x[0], x[1][1]))) # (article_id, (user_id, {user_profile}))
                   .join(articles_profile.groupByKey().mapValues(dict)) # (article_id, ((user_id, {user_profile}), {article_profile}))
                   .map(lambda x: (x[1][0][0], (x[0], cosine_similarity(x[1][0][1], x[1][1])))) # (user_id, (article_id, actual_sim))
                   .filter(lambda x: x[1][1] > 0.0))

print(f"Number of recommendations with actual cosine similarity: {recommendations.count()}")
recommendations.take(10)


Number of recommendations with actual cosine similarity: 6003516


[(66455289,
  ('nyt://article/a72bd818-ae7c-578f-a666-0989b959ca2b', 0.0944911182523068)),
 (75812328,
  ('nyt://article/a72bd818-ae7c-578f-a666-0989b959ca2b', 0.2988071523335984)),
 (48674790,
  ('nyt://article/a72bd818-ae7c-578f-a666-0989b959ca2b', 0.14433756729740646)),
 (26857980,
  ('nyt://article/a72bd818-ae7c-578f-a666-0989b959ca2b', 0.22360679774997896)),
 (30115161,
  ('nyt://article/a72bd818-ae7c-578f-a666-0989b959ca2b', 0.12126781251816648)),
 (58955814,
  ('nyt://article/a72bd818-ae7c-578f-a666-0989b959ca2b', 0.17541160386140586)),
 (78777270,
  ('nyt://article/a72bd818-ae7c-578f-a666-0989b959ca2b', 0.26726124191242434)),
 (66297033,
  ('nyt://article/a72bd818-ae7c-578f-a666-0989b959ca2b', 0.10910894511799617)),
 (78679404,
  ('nyt://article/a72bd818-ae7c-578f-a666-0989b959ca2b', 0.30831320798910367)),
 (58635846,
  ('nyt://article/a72bd818-ae7c-578f-a666-0989b959ca2b', 0.17677669529663687))]

# Content-Based Recommendations

In [16]:
import heapq

user_recommendations = (recommendations
                        .map(lambda x: (x[0], (x[1][0], round(x[1][1], 5))))
                        .groupByKey()
                        .mapValues(lambda iter: heapq.nlargest(RECOMMENDED_ARTICLES, iter, key=lambda x: x[1])))

print(f"Number of users with at least one recommendation: {user_recommendations.count()}/{users.count()}")
user_recommendations.take(10)


Number of users with at least one recommendation: 83987/91437


[(48674790,
  [('nyt://article/4e7f2a77-8938-57f2-b15e-647f5a31042c', 0.25),
   ('nyt://article/d614dc23-deba-5932-ae91-e8a399b1e9e3', 0.2357),
   ('nyt://article/ef5a9533-6c71-5b61-b5f0-85c763d01a1b', 0.21651),
   ('nyt://article/3ad369b1-dedf-54ba-a9c0-3251ff021580', 0.21651),
   ('nyt://article/1e4558c1-85a0-59cb-8753-3dcfc65d2a11', 0.21082),
   ('nyt://article/c34a9ea4-b186-5952-8d05-8736ec3277de', 0.20412),
   ('nyt://article/0847618f-a9ac-5195-9193-6a13b75108c2', 0.19365),
   ('nyt://article/a6e7504d-2aa4-5116-ad4b-37a4d424c785', 0.19365),
   ('nyt://article/991449a1-98ba-5551-8716-91587e925d4c', 0.18257),
   ('nyt://article/0b1330dc-d580-5285-81a2-a08c507f809b', 0.18257)]),
 (26857980,
  [('nyt://article/deaed1ea-7c1a-57d7-a102-c9f4d85ff93b', 0.7303),
   ('nyt://article/f31f3b3b-49ca-5774-b8d1-81dca9e2687a', 0.7303),
   ('nyt://article/c08db3d1-6d65-59af-a255-d8a61ab09348', 0.7303),
   ('nyt://article/28ed542c-a4d8-5e69-9356-49929b142471', 0.7303),
   ('nyt://article/b9049bfc-85

## Human Evaluation

In [ ]:
users.takeSample(False, 10)


[80613460,
 68862237,
 61167964,
 83406317,
 58730684,
 76577034,
 43325423,
 20681453,
 52729395,
 67242185]

In [20]:
# USER = 65969500 # bad results
# USER = 31496310 # ok?
USER = 61167964

read = (comments
        .filter(lambda x: x[1] == USER)
        .join(full_articles.rdd.map(lambda x: (x['uniqueID'], x.asDict()))) # (article_id, (user_id, article_info))
        .map(lambda x: x[1][1]) # (article_info)
        .collect())

reccs = (user_recommendations
         .filter(lambda x: x[0] == USER)
         .map(lambda x: x[1])
         .flatMap(lambda x: x)
         .join(full_articles.rdd.map(lambda x: (x['uniqueID'], x.asDict()))) # (article_id, article_info)
         .map(lambda x: (x[1][1], x[1][0])) # (article_info, similarity)
         .collect())

print("--- ARTICLES READ ---")
for r in read:
    print(f"{r['headline']}\n\t{r['newsdesk']} - {r['section']} - {r['subsection']}\n\t{r['keywords']}")

print("--- ARTICLES RECOMMENDED ---")
for r, s in sorted(reccs, key=lambda x: x[1], reverse=True):
    print(f"{round(s, 5)} {r['headline']}\n\t{r['newsdesk']} - {r['section']} - {r['subsection']}\n\t{r['keywords']}")


--- ARTICLES READ ---
Suleimani Died as He Had Killed
	OpEd - Opinion - None
	['Iran', 'Suleimani, Qassim', 'Defense and Military Forces', 'Quds Force', 'United States International Relations', 'United States Defense and Military Forces', 'Trump, Donald J', 'Baghdad (Iraq)', 'Lebanon', 'Middle East', 'Filkins, Dexter', 'Targeted Killings']
Trump Kills Iran’s Most Overrated Warrior
	OpEd - Opinion - None
	['Iran', 'Suleimani, Qassim', 'Quds Force', 'Defense and Military Forces', 'Trump, Donald J', 'United States International Relations', 'United States Defense and Military Forces', 'Targeted Killings']
Pelosi Is Prepared to Send Impeachment Articles to Senate, Just Not Yet
	Washington - U.S. - Politics
	['Pelosi, Nancy', 'House of Representatives', 'McConnell, Mitch', 'Senate', 'Democratic Party', 'Republican Party', 'Impeachment', 'Trump, Donald J', 'Trump-Ukraine Whistle-blower Complaint and Impeachment Inquiry', 'United States Politics and Government']
Prince Harry’s Real Declaration